In [5]:
!pip install pandas numpy matplotlib seaborn ipywidgets scipy

In [14]:
# 1. Импорт библиотек
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from glob import glob
import re
import os
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Настройка отображения
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
%matplotlib inline
plt.style.use('seaborn-v0_8-darkgrid')

# 2. Функция для парсинга информации о задаче из имени файла
def parse_task_info(filename):
    """
    Парсит информацию о задаче из имени файла
    Пример: ../../dataset_directory_big/20_20/2349008_20_20_0.300000_0.300000_0.300000_1000_100_.txt
    Формат: dataset_id_m_n_a_b_c_D_S_.txt
    """
    basename = os.path.basename(filename).replace('.txt', '')
    parts = basename.split('_')
    
    info = {
        'file_path': filename,
        'filename': basename,
        'seed': parts[0],
        'cnt_tools': int(parts[1]),  # вероятно, число машин
        'interval_per_tool': int(parts[2]),  # вероятно, число работ
        'operation_density': float(parts[3]),  # параметр 1
        'edge_density': float(parts[4]),  # параметр 2
        'tool_on_operation_density': float(parts[5]),  # параметр 3
        'min_interval_dur': int(parts[6]),  # вероятно, максимальная длительность
        'min_spacer_dur': int(parts[7]),  # вероятно, seed или что-то подобное
        'full_filename': filename
    }
    return info

# 3. Чтение всех CSV файлов
# Предполагаем, что файлы находятся в текущей директории
csv_files = glob('*.csv')  # или укажите конкретный путь

all_data = []

for file in csv_files:
    try:
        # Читаем CSV файл
        df = pd.read_csv(file, sep=';')
        
        # Парсим информацию о задачах из первого столбца
        task_info = []
        for path in df.iloc[:, 0]:
            info = parse_task_info(path)
            task_info.append(info)
        
        # Создаем DataFrame с информацией о задачах
        info_df = pd.DataFrame(task_info)
        
        # Объединяем информацию о задачах с результатами
        # Первый столбец содержит пути, остальные - результаты алгоритмов
        results_df = df.iloc[:, 1:].copy()
        results_df.columns = df.columns[1:]  # сохраняем имена алгоритмов
        
        # Объединяем
        combined_df = pd.concat([info_df.reset_index(drop=True), 
                                results_df.reset_index(drop=True)], axis=1)
        
        # Добавляем имя исходного файла с результатами
        combined_df['results_file'] = os.path.basename(file)
        
        all_data.append(combined_df)
        print(f"Обработан файл: {file}, строк: {len(combined_df)}")
        
    except Exception as e:
        print(f"Ошибка при обработке файла {file}: {str(e)}")

# 4. Объединяем все данные в один DataFrame
if all_data:
    full_df = pd.concat(all_data, ignore_index=True)
    print(f"\nВсего строк в объединенном датасете: {len(full_df)}")
    print(f"Колонки: {list(full_df.columns)}\n")
else:
    print("Нет данных для обработки!")
    full_df = pd.DataFrame()

# Покажем первые несколько строк
if not full_df.empty:
    display(full_df)

Обработан файл: dataset_table_45_20.csv, строк: 540
Обработан файл: dataset_table_20_70.csv, строк: 540
Обработан файл: dataset_table_35_70.csv, строк: 540
Обработан файл: dataset_table_35_40.csv, строк: 540
Обработан файл: dataset_table_20_20.csv, строк: 540
Обработан файл: dataset_table_20_40.csv, строк: 540
Обработан файл: dataset_table_45_25.csv, строк: 540
Обработан файл: dataset_table_20_25.csv, строк: 540
Обработан файл: dataset_table_45_40.csv, строк: 540
Обработан файл: dataset_table_35_20.csv, строк: 540
Обработан файл: dataset_table_35_25.csv, строк: 540
Обработан файл: dataset_table_45_70.csv, строк: 540

Всего строк в объединенном датасете: 6480
Колонки: ['file_path', 'filename', 'seed', 'cnt_tools', 'interval_per_tool', 'operation_density', 'edge_density', 'tool_on_operation_density', 'min_interval_dur', 'min_spacer_dur', 'full_filename', 'Dummy', 'Directive', 'Stoppable', 'Robin', 'FineSorter', 'Dependent', 'MDummy6_2', 'MDirective6_2', 'MStoppable6_2', 'MRobin6_2', 'MFi

,file_path,filename,seed,cnt_tools,interval_per_tool,operation_density,edge_density,tool_on_operation_density,min_interval_dur,min_spacer_dur,full_filename,Dummy,Directive,Stoppable,Robin,FineSorter,Dependent,MDummy6_2,MDirective6_2,MStoppable6_2,MRobin6_2,MFineSorter6_2,MDependent6_2,MDummy3_4,MDirective3_4,MStoppable3_4,MRobin3_4,MFineSorter3_4,MDependent3_4,MDummy2_8,MDirective2_8,MStoppable2_8,MRobin2_8,MFineSorter2_8,MDependent2_8,Aggregator,results_file
0,../../dataset_directory_big/45_20/2349008_45_2...,2349008_45_20_0.300000_0.300000_0.300000_1000_...,2349008,45,20,0.30,0.30,0.30,1000,100,../../dataset_directory_big/45_20/2349008_45_2...,81.3233,54.2312,81.3233,54.2312,54.2312,440.441,97.8395,54.2312,54.2312,54.2312,54.2312,440.441,187.727,54.2312,144.119,54.2312,54.2312,440.441,54.2312,54.2312,54.2312,54.2312,54.2312,440.441,97.8395,dataset_table_45_20.csv
1,../../dataset_directory_big/45_20/2349008_45_2...,2349008_45_20_0.300000_0.300000_0.300000_1000_...,2349008,45,20,0.30,0.30,0.30,1000,500,../../dataset_directory_big/45_20/2349008_45_2...,329.2780,273.3080,342.0620,339.7700,283.4490,456.489,350.8550,273.3080,210.6400,287.7790,283.4490,437.279,241.527,273.3080,241.527,286.3990,283.4490,478.077,283.4490,273.3080,252.5620,339.5140,283.4490,437.279,396.2710,dataset_table_45_20.csv
2,../../dataset_directory_big/45_20/2349008_45_2...,2349008_45_20_0.300000_0.300000_0.300000_200_100_,2349008,45,20,0.30,0.30,0.30,200,100,../../dataset_directory_big/45_20/2349008_45_2...,114.8080,114.4600,114.8080,161.9840,108.0560,110.190,114.9150,114.4600,108.0560,108.0560,108.0560,108.056,161.480,114.4600,108.056,108.0560,108.0560,108.056,114.4600,114.4600,108.0560,108.0560,108.0560,108.056,108.0560,dataset_table_45_20.csv
3,../../dataset_directory_big/45_20/2349008_45_2...,2349008_45_20_0.300000_0.300000_0.300000_200_500_,2349008,45,20,0.30,0.30,0.30,200,500,../../dataset_directory_big/45_20/2349008_45_2...,270.4950,293.2730,258.6000,547.6560,331.3260,302.120,353.3720,293.2730,231.5580,172.8150,331.3260,350.644,299.423,293.2730,270.777,210.4660,331.3260,300.356,310.8660,293.2730,267.9570,409.1770,331.3260,326.329,278.9660,dataset_table_45_20.csv
4,../../dataset_directory_big/45_20/2349008_45_2...,2349008_45_20_0.300000_0.300000_0.450000_1000_...,2349008,45,20,0.30,0.30,0.45,1000,100,../../dataset_directory_big/45_20/2349008_45_2...,255.4860,238.7850,299.3210,257.3800,246.5340,231.781,224.5490,238.7850,257.1360,219.2380,246.5340,241.422,224.637,238.7850,236.637,249.1020,246.5340,241.422,224.6370,238.7850,234.9340,224.6370,246.5340,235.124,227.8780,dataset_table_45_20.csv
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6475,../../dataset_directory_big/45_70/68123_45_70_...,68123_45_70_0.690000_0.690000_0.450000_200_500_,68123,45,70,0.69,0.69,0.45,200,500,../../dataset_directory_big/45_70/68123_45_70_...,5333.8600,5024.9700,5210.5900,6137.2900,3121.8900,5816.710,3441.7600,5024.9700,3007.7100,4341.6800,3121.8900,5807.630,3632.520,5024.9700,3726.120,4569.7300,3121.8900,6612.380,3196.3300,5024.9700,3229.2000,5781.9600,3121.8900,6079.700,2920.8300,dataset_table_45_70.csv
6476,../../dataset_directory_big/45_70/68123_45_70_...,68123_45_70_0.690000_0.690000_0.690000_1000_100_,68123,45,70,0.69,0.69,0.69,1000,100,../../dataset_directory_big/45_70/68123_45_70_...,3114.0800,2449.8000,2683.5800,2883.4500,2363.3400,3099.410,2014.4300,2449.8000,2135.0200,2678.0300,2363.3400,2710.800,2272.100,2449.8000,1974.140,2180.9600,2363.3400,2948.700,2360.9200,2449.8000,2222.6000,2693.9100,2363.3400,4563.940,2011.4500,dataset_table_45_70.csv
6477,../../dataset_directory_big/45_70/68123_45_70_...,68123_45_70_0.690000_0.690000_0.690000_1000_500_,68123,45,70,0.69,0.69,0.69,1000,500,../../dataset_directory_big/45_70/68123_45_70_...,3028.4200,3667.5300,2704.0700,2658.9500,2367.7700,2952.610,1621.6700,3667.5300,2114.8900,2473.0500,2367.7700,2628.210,2219.010,3667.5300

In [7]:
# 5. Обработка данных
if not full_df.empty:
    # Проверяем наличие столбца Dummy
    if 'Dummy' in full_df.columns:
        print("Статистика по столбцу Dummy (базовый алгоритм):")
        print(full_df['Dummy'].describe())
        
        # Удаляем строки, где Dummy = 0 или отрицательный (выбросы)
        initial_count = len(full_df)
        full_df = full_df[full_df['Dummy'] > 0]
        removed_count = initial_count - len(full_df)
        print(f"\nУдалено строк с Dummy <= 0: {removed_count}")
        print(f"Осталось строк: {len(full_df)}")
        # Создаем список алгоритмов для сравнения (исключая Dummy и информационные колонки)
        info_columns = ['file_path', 'filename', 'seed', 'cnt_tools', 'interval_per_tool', 
                        'operation_density', 'edge_density', 'tool_on_operation_density', 
                        'min_interval_dur', 'min_spacer_dur', 'full_filename', 'results_file', "best_algorithm"]
        
        algorithm_columns = [col for col in full_df.columns 
                           if col not in info_columns and col != 'Dummy']
        
        print(f"\nАлгоритмы для анализа ({len(algorithm_columns)}):")
        print(algorithm_columns)
        
        # 6. Вычисляем относительное качество (Dummy / Algorithm)
        # Меньшее значение Dummy/Algorithm означает лучший алгоритм
        for algo in algorithm_columns:
            if algo in full_df.columns:
                full_df[f'{algo}_rel_to_Dummy'] = full_df[algo] / full_df['Dummy']
        
        # Добавляем флаг, является ли алгоритм лучше Dummy
        for algo in algorithm_columns:
            if algo in full_df.columns:
                full_df[f'{algo}_better_than_Dummy'] = full_df[f'{algo}_rel_to_Dummy'] < 1
        
        # 7. Находим лучший алгоритм для каждой задачи
        def find_best_algorithm(row, algorithms):
            best_algo = None
            best_value = None
            
            for algo in algorithms:
                if algo in row and pd.notnull(row[algo]):
                    # Ищем минимальное значение (лучший результат)
                    if best_value is None or row[algo] < best_value:
                        best_value = row[algo]
                        best_algo = algo
            
            return pd.Series([best_algo, best_value], index=['best_algorithm', 'best_value'])
        
        # Применяем функцию
        best_results = full_df.apply(
            lambda row: find_best_algorithm(row, algorithm_columns), 
            axis=1
        )
        
        full_df = pd.concat([full_df, best_results], axis=1)
        
        # 8. Вычисляем, насколько лучший алгоритм лучше Dummy
        full_df['best_improvement_over_Dummy'] = full_df['best_value'] / full_df['Dummy']
        
        print("\nСтатистика по улучшению лучшего алгоритма над Dummy:")
        print(full_df['best_improvement_over_Dummy'].describe())
        
        # Покажем несколько строк с результатами
        display(full_df[['seed', 'cnt_tools', 'interval_per_tool', 'min_interval_dur', 'min_spacer_dur', 'Dummy', 
                        'best_algorithm', 'best_value', 'best_improvement_over_Dummy']].head(10))

Статистика по столбцу Dummy (базовый алгоритм):
count     6480.000000
mean      1464.610741
std       2056.631023
min          0.000000
25%         70.312750
50%        437.572500
75%       2193.305000
max      13181.000000
Name: Dummy, dtype: float64

Удалено строк с Dummy <= 0: 665
Осталось строк: 5815

Алгоритмы для анализа (24):
['Directive', 'Stoppable', 'Robin', 'FineSorter', 'Dependent', 'MDummy6_2', 'MDirective6_2', 'MStoppable6_2', 'MRobin6_2', 'MFineSorter6_2', 'MDependent6_2', 'MDummy3_4', 'MDirective3_4', 'MStoppable3_4', 'MRobin3_4', 'MFineSorter3_4', 'MDependent3_4', 'MDummy2_8', 'MDirective2_8', 'MStoppable2_8', 'MRobin2_8', 'MFineSorter2_8', 'MDependent2_8', 'Aggregator']

Статистика по улучшению лучшего алгоритма над Dummy:
count    5815.000000
mean        0.419299
std         0.288050
min         0.000000
25%         0.137048
50%         0.464911
75%         0.648812
max         1.571149
Name: best_improvement_over_Dummy, dtype: float64


,seed,cnt_tools,interval_per_tool,min_interval_dur,min_spacer_dur,Dummy,best_algorithm,best_value,best_improvement_over_Dummy
0,2349008,45,20,1000,100,81.32330,Directive,54.2312,0.666859
1,2349008,45,20,1000,500,329.27800,MStoppable6_2,210.6400,0.639703
2,2349008,45,20,200,100,114.80800,FineSorter,108.0560,0.941189
3,2349008,45,20,200,500,270.49500,MRobin6_2,172.8150,0.638884
4,2349008,45,20,1000,100,255.48600,MRobin6_2,219.2380,0.858121
5,2349008,45,20,1000,500,308.40500,MStoppable2_8,214.9950,0.697119
6,2349008,45,20,200,100,41.53770,MDependent6_2,28.1380,0.677409
7,2349008,45,20,200,500,200.02200,FineSorter,136.7500,0.683675
8,2349008,45,20,1000,100,54.62290,MDummy6_2,39.7145,0.727067
9,2349008,45,20,1000,500,7.68858,Robin,1.9944,0.259398


In [13]:
# 9. Анализ и визуализация
if not full_df.empty:
    # Создаем интерактивный выбор характеристик
    from ipywidgets import interact, interactive, fixed, interact_manual
    import ipywidgets as widgets
    
    # Функция для фильтрации данных по характеристикам
    def filter_and_analyze(m_min=None, m_max=None, n_min=None, n_max=None, 
                          D_min=None, D_max=None, a_min=None, a_max=None):
        # Создаем копию данных
        filtered_df = full_df.copy()
        
        # Применяем фильтры
        if m_min is not None:
            filtered_df = filtered_df[filtered_df['cnt_tools'] >= m_min]
        if m_max is not None:
            filtered_df = filtered_df[filtered_df['cnt_tools'] <= m_max]
        if n_min is not None:
            filtered_df = filtered_df[filtered_df['interval_per_tool'] >= n_min]
        if n_max is not None:
            filtered_df = filtered_df[filtered_df['interval_per_tool'] <= n_max]
        if D_min is not None:
            filtered_df = filtered_df[filtered_df['min_interval_dur'] >= D_min]
        if D_max is not None:
            filtered_df = filtered_df[filtered_df['min_interval_dur'] <= D_max]
        if a_min is not None:
            filtered_df = filtered_df[filtered_df['operation_density'] >= a_min]
        if a_max is not None:
            filtered_df = filtered_df[filtered_df['operation_density'] <= a_max]
        
        print(f"Отфильтровано задач: {len(filtered_df)}")
        
        if len(filtered_df) == 0:
            print("Нет данных, соответствующих фильтрам")
            return
        
        # Анализ лучших алгоритмов
        best_algo_counts = filtered_df['best_algorithm'].value_counts()
        
        # Создаем визуализацию
        fig, axes = plt.subplots(2, 3, figsize=(18, 12))
        
        # 1. Распределение лучших алгоритмов
        ax1 = axes[0, 0]
        best_algo_counts.plot(kind='bar', ax=ax1, color='skyblue')
        ax1.set_title('Частота лучших алгоритмов')
        ax1.set_xlabel('Алгоритм')
        ax1.set_ylabel('Количество задач')
        ax1.tick_params(axis='x', rotation=45)
        
        # 2. Улучшение над Dummy
        ax2 = axes[0, 1]
        filtered_df['best_improvement_over_Dummy'].hist(bins=30, ax=ax2, 
                                                       edgecolor='black', alpha=0.7)
        ax2.axvline(x=1, color='red', linestyle='--', label='Равен Dummy')
        ax2.set_title('Распределение улучшения лучшего алгоритма над Dummy')
        ax2.set_xlabel('Во сколько раз лучше Dummy')
        ax2.set_ylabel('Количество задач')
        ax2.legend()
        
        # 3. Boxplot улучшений по алгоритмам
        ax3 = axes[0, 2]
        # Собираем данные для boxplot
        boxplot_data = []
        boxplot_labels = []
        
        top_algos = best_algo_counts.head(10).index.tolist()
        for algo in top_algos:
            rel_col = f'{algo}_rel_to_Dummy'
            if rel_col in filtered_df.columns:
                boxplot_data.append(filtered_df[rel_col].dropna())
                boxplot_labels.append(algo)
        
        if boxplot_data:
            ax3.boxplot(boxplot_data, labels=boxplot_labels)
            ax3.axhline(y=1, color='red', linestyle='--', alpha=0.5)
            ax3.set_title('Относительная эффективность алгоритмов (vs Dummy)')
            ax3.set_ylabel('Algorithm / Dummy')
            ax3.tick_params(axis='x', rotation=45)
        
        # 4. Heatmap зависимости от m и n
        ax4 = axes[1, 0]
        if len(filtered_df) > 1:
            pivot_table = filtered_df.pivot_table(
                values='best_improvement_over_Dummy',
                index='cnt_tools',
                columns='interval_per_tool',
                aggfunc='mean'
            )
            if not pivot_table.empty:
                sns.heatmap(pivot_table, annot=True, fmt='.2f', cmap='YlOrRd', 
                           ax=ax4, cbar_kws={'label': 'Среднее улучшение'})
                ax4.set_title('Среднее улучшение по m и n')
        
        # 5. Лучшие алгоритмы по размеру задачи
        ax5 = axes[1, 1]
        problem_sizes = filtered_df['cnt_tools'] * filtered_df['interval_per_tool']
        filtered_df['problem_size'] = problem_sizes
        
        # Группируем по размеру задачи и находим лучший алгоритм
        size_groups = filtered_df.groupby('problem_size')['best_algorithm'].agg(
            lambda x: x.mode()[0] if not x.mode().empty else None
        ).dropna()
        
        if not size_groups.empty:
            size_groups.value_counts().head(10).plot(
                kind='pie', ax=ax5, autopct='%1.1f%%'
            )
            ax5.set_title('Лучшие алгоритмы по размеру задачи')
            ax5.set_ylabel('')
        
        # 6. Топ-5 алгоритмов по средней эффективности
        ax6 = axes[1, 2]
        algo_efficiency = []
        for algo in algorithm_columns:
            rel_col = f'{algo}_rel_to_Dummy'
            if rel_col in filtered_df.columns:
                mean_eff = filtered_df[rel_col].mean()
                algo_efficiency.append((algo, mean_eff))
        
        algo_efficiency.sort(key=lambda x: x[1], reverse=False)
        top_algos_eff = algo_efficiency[:5]
        
        if top_algos_eff:
            algos, effs = zip(*top_algos_eff)
            ax6.barh(range(len(algos)), effs, color='lightgreen')
            ax6.set_yticks(range(len(algos)))
            ax6.set_yticklabels(algos)
            ax6.set_xlabel('Среднее Algorithm/Dummy')
            ax6.set_title('Топ-5 алгоритмов по эффективности')
            ax6.axvline(x=1, color='red', linestyle='--', alpha=0.5)
        
        plt.tight_layout()
        plt.show()
        
        # Вывод статистики
        print("\n=== СТАТИСТИКА ПО ОТФИЛЬТРОВАННЫМ ДАННЫМ ===")
        print(f"Всего задач: {len(filtered_df)}")
        print(f"\nЛучшие алгоритмы (топ-10):")
        print(best_algo_counts.head(10))
        
        print(f"\nСреднее улучшение над Dummy: {filtered_df['best_improvement_over_Dummy'].mean():.3f}")
        print(f"Медианное улучшение: {filtered_df['best_improvement_over_Dummy'].median():.3f}")
        print(f"Максимальное улучшение: {filtered_df['best_improvement_over_Dummy'].max():.3f}")
        
        # Процент задач, где каждый алгоритм был лучшим
        print("\nПроцент задач, где алгоритм был лучшим:")
        for algo, count in best_algo_counts.head(5).items():
            percentage = (count / len(filtered_df)) * 100
            mean_improvement = filtered_df.loc[
                filtered_df['best_algorithm'] == algo, 'best_improvement_over_Dummy'
            ].mean()
            print(f"{algo}: {percentage:.1f}% задач, среднее улучшение: {mean_improvement:.3f}")
    
    # Создаем виджеты для фильтрации
    m_values = sorted(full_df['cnt_tools'].unique()) if 'cnt_tools' in full_df.columns else [0]
    n_values = sorted(full_df['interval_per_tool'].unique()) if 'interval_per_tool' in full_df.columns else [0]
    D_values = sorted(full_df['min_interval_dur'].unique()) if 'min_interval_dur' in full_df.columns else [0]
    a_values = sorted(full_df['operation_density'].unique()) if 'operation_density' in full_df.columns else [0]
    
    # Интерактивный интерфейс
    interact(
        filter_and_analyze,
        m_min=widgets.IntSlider(min=min(m_values), max=max(m_values), 
                               step=1, value=min(m_values), description='cnt_tools min'),
        m_max=widgets.IntSlider(min=min(m_values), max=max(m_values), 
                               step=1, value=max(m_values), description='cnt_tools max'),
        n_min=widgets.IntSlider(min=min(n_values), max=max(n_values), 
                               step=1, value=min(n_values), description='interval_per_tool min'),
        n_max=widgets.IntSlider(min=min(n_values), max=max(n_values), 
                               step=1, value=max(n_values), description='interval_per_tool max'),
        D_min=widgets.IntSlider(min=min(D_values), max=max(D_values), 
                               step=100, value=min(D_values), description='min_interval_dur min'),
        D_max=widgets.IntSlider(min=min(D_values), max=max(D_values), 
                               step=100, value=max(D_values), description='min_interval_dur max'),
        a_min=widgets.FloatSlider(min=min(a_values), max=max(a_values), 
                                 step=0.05, value=min(a_values), description='operation_density min'),
        a_max=widgets.FloatSlider(min=min(a_values), max=max(a_values), 
                                 step=0.05, value=max(a_values), description='operation_density max')
    )

interactive(children=(IntSlider(value=20, description='cnt_tools min', max=45, min=20), IntSlider(value=45, de…

In [9]:
# 10. Детальный анализ конкретного алгоритма
if not full_df.empty:
    def analyze_algorithm(algorithm_name):
        if algorithm_name not in full_df.columns:
            print(f"Алгоритм {algorithm_name} не найден в данных")
            return
        
        print(f"=== АНАЛИЗ АЛГОРИТМА: {algorithm_name} ===")
        
        # Статистика по значениям алгоритма
        print(f"\nСтатистика по значениям алгоритма:")
        print(full_df[algorithm_name].describe())
        
        # Относительная эффективность
        rel_col = f'{algorithm_name}_rel_to_Dummy'
        if rel_col in full_df.columns:
            print(f"\nОтносительная эффективность ({algorithm_name} / Dummy):")
            print(f"Среднее: {full_df[rel_col].mean():.3f}")
            print(f"Медиана: {full_df[rel_col].median():.3f}")
            print(f"Минимум: {full_df[rel_col].min():.3f}")
            print(f"Максимум: {full_df[rel_col].max():.3f}")
            
            # Процент задач, где алгоритм лучше Dummy
            better_col = f'{algorithm_name}_better_than_Dummy'
            if better_col in full_df.columns:
                better_percentage = full_df[better_col].mean() * 100
                print(f"\nПроцент задач, где алгоритм лучше Dummy: {better_percentage:.1f}%")
        
        # Визуализация
        fig, axes = plt.subplots(1, 3, figsize=(15, 5))
        
        # 1. Распределение значений
        axes[0].hist(full_df[algorithm_name], bins=30, edgecolor='black', alpha=0.7)
        axes[0].set_title(f'Распределение значений {algorithm_name}')
        axes[0].set_xlabel('Значение')
        axes[0].set_ylabel('Частота')
        
        # 2. Сравнение с Dummy
        if rel_col in full_df.columns:
            axes[1].hist(full_df[rel_col], bins=30, edgecolor='black', alpha=0.7)
            axes[1].axvline(x=1, color='red', linestyle='--', label='Равен Dummy')
            axes[1].set_title(f'Относительная эффективность {algorithm_name}')
            axes[1].set_xlabel('Dummy / Algorithm')
            axes[1].set_ylabel('Частота')
            axes[1].legend()
        
        # 3. Зависимость от размера задачи
        axes[2].scatter(full_df['cnt_tools'] * full_df['interval_per_tool'], full_df[algorithm_name], 
                       alpha=0.5, s=20)
        axes[2].set_title(f'{algorithm_name} vs размер задачи')
        axes[2].set_xlabel('m × n')
        axes[2].set_ylabel('Значение алгоритма')
        
        plt.tight_layout()
        plt.show()
    
    # Выбор алгоритма для анализа
    algorithm_selector = widgets.Dropdown(
        options=algorithm_columns,
        value=algorithm_columns[0] if algorithm_columns else None,
        description='Алгоритм:',
        disabled=False,
    )
    
    interact(analyze_algorithm, algorithm_name=algorithm_selector)

interactive(children=(Dropdown(description='Алгоритм:', options=('Directive', 'Stoppable', 'Robin', 'FineSorte…